In [ ]:
from PIL import Image
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from pathlib import Path
from collections import Counter
from matplotlib import colors

In [ ]:
import re
import os
import hashlib
import numpy as np
import pandas as pd
import torch as to
import matplotlib.pyplot as plt
import torch.nn as nn

In [ ]:
DATA_DIR = Path("data")
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4
PATIENCE = 4         
SEED = 42
DEVICE = "cuda" if to.cuda.is_available() else "cpu"
CLASSES = {"non-COVID": 0, "COVID": 1}
EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

In [ ]:
to.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
rows = []
for cls_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for p in cls_dir.rglob("*"):
        if p.suffix.lower() not in EXTS:
            continue
        row = {"path": str(p), "cls": cls_dir.name, "name": p.name, "ok": True}
        try:
            with Image.open(p) as im:
                im.verify()                     
            with Image.open(p) as im:
                arr = np.asarray(im.convert("L"), dtype=np.float32)
                row.update(
                    width=im.width, height=im.height, mode=im.mode,
                    mean=arr.mean(), std=arr.std(),
                    min=arr.min(), max=arr.max(),
                    black_frac=(arr < 10).mean(),   
                )
            row["md5"] = hashlib.md5(p.read_bytes()).hexdigest()
        except Exception as e:
            row.update(ok=False, error=str(e))
        rows.append(row)
 
df = pd.DataFrame(rows)
print(f"\nTotal images found: {len(df)}")

In [ ]:
print("\n=== Class balance ===")
print(df.cls.value_counts().to_string()) # df["cls"].value_counts().to_string() also works

In [ ]:
bad = df[~df.ok]
print(f"\n=== Corrupt/unreadable files: {len(bad)} ===")
if len(bad):
    print(bad[["path", "error"]].head(10).to_string())
df = df[df.ok].copy()

In [ ]:
print("\n=== Image sizes (top 10) ===")
sizes = Counter(zip(df.width, df.height))
for (w, h), n in sizes.most_common(10):
    print(f"{w}x{h}: {n}")
print(f"Distinct sizes: {len(sizes)}")
 
print("\n=== Color modes ===")
print(df["mode"].value_counts().to_string())
 
print("\n=== Size per class (mean width x height) ===")
print(df.groupby("cls")[["width", "height"]].mean().round(0).to_string())